# 🚀 Train LoRA Qwen2.5-1.5B-Instruct cho Tool-Calling
**Dataset:** Dataset Speech-MASSIVE vừa sinh  
**Thư viện:** `unsloth` (nhanh gấp 2x và tiết kiệm VRAM)  
**GPU:** T4 (16GB VRAM)

In [ ]:
# @title 1. Cài đặt Unsloth & Thư viện
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
# @title 2. Nạp Model Qwen2.5-1.5B-Instruct ở 4-bit
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Có thể tăng nếu RAM cho phép
dtype = None # Auto tự nhận diện Float16 cho T4
load_in_4bit = True # Bắt buộc dùng 4bit để giảm VRAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
# @title 3. Gắn LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Cỡ LoRA (16, 32, 64, 128)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"], # Target toàn bộ MLP & Attention
    lora_alpha = 16,
    lora_dropout = 0, # Unsloth tối ưu nếu = 0
    bias = "none",
    use_gradient_checkpointing = "unsloth", # True hoặc "unsloth" cho long context
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

print("✅ Đã gắn LoRA!")

In [ ]:
# @title 4. Chuẩn bị Dataset
from datasets import load_dataset

# THAY BẰNG TÊN DATASET CỦA BẠN TRÊN HUGGINGFACE
DATASET_ID = "YOUR_USERNAME/speech-massive-vie-tool-calling" 
dataset = load_dataset(DATASET_ID, split="train")

def format_chat_template(example):
    messages = [
        {"role": "system", "content": example["instruction"]},
        {"role": "user", "content": example["input"]},
        {"role": "assistant", "content": example["output"]}
    ]
    # tokenizer của Qwen hỗ trợ apply_chat_template sẵn
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
    return {"text": text}

dataset = dataset.map(format_chat_template)
print(dataset[0]["text"])

In [ ]:
# @title 5. Cấu hình & Bắt đầu Train
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Pack nhiều câu ngắn thành 1 seq để train nhanh hơn
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 1, # Chạy 1 epoch để test trước
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

In [ ]:
# @title 6. Lưu Model (Chỉ lưu Weights LoRA)
model.save_pretrained("lora_model") # Lưu Local
tokenizer.save_pretrained("lora_model")

# Nếu muốn push thẳng lên HuggingFace Hub, bỏ comment 2 dòng dưới:
# model.push_to_hub("YOUR_USERNAME/qwen-1.5b-tool-calling-lora", token = "hf_...") 
# tokenizer.push_to_hub("YOUR_USERNAME/qwen-1.5b-tool-calling-lora", token = "hf_...")
print("✅ Saved LoRA weights to `lora_model`")

In [ ]:
# @title 7. Inference Test (Chạy thử Model sau khi train)
FastLanguageModel.for_inference(model) # Bật chế độ chạy cực nhanh của Unsloth

test_messages = [
    {"role": "system", "content": "Phản hồi câu nói dưới dạng tool call hoặc trả lời tự nhiên.\nÝ định: play_music\nBối cảnh: play"},
    {"role": "user", "content": "mở bài hát ưng quá chừng của amee"}
]
inputs = tokenizer.apply_chat_template(
    test_messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt=True)

print("\n--- KẾT QUẢ SINH TỪ MODEL ĐÃ TRAIN ---")
_ = model.generate(input_ids=inputs, streamer=text_streamer, max_new_tokens=128, temperature=0.1)